# Tutorial V - Production Planning in Breweries

Applied Optimization with Julia

# 1. Modelling the CLSP

Implement the CLSP from the lecture in Julia. Before we start, let’s
load the necessary packages and data.

In [1]:
using JuMP, HiGHS
using CSV
using DelimitedFiles
using DataFrames
using Plots
using StatsPlots

> **Tip**
>
> If you haven’t installed the packages yet, you can do so by running
> `using Pkg` first and then `Pkg.add("JuMP")`, `Pkg.add("HiGHS")`,
> `Pkg.add("DataFrames")`, `Pkg.add("Plots")`, and
> `Pkg.add("StatsPlots")`.

Now, let’s load the data. The weekly demand in bottles $d_{i,t}$, the
available time at the bottling plant in hours $a_t$, the time required
to bottle each beer in hours $b_i$, and the setup time in hours $g_i$
are provided as CSV files.

In [1]:
# Get the directory of the current file
file_directory = "$(@__DIR__)/data"

# Load the data about the available time at the bottling plant
availableTime = CSV.read("$file_directory/availabletime.csv", DataFrame)
println("Number of periods: $(nrow(availableTime))")
println("First 5 rows of available time per period:")
println(availableTime[1:5, :])

Number of periods: 27
First 5 rows of available time per period:
5×2 DataFrame
 Row │ period   available_capacity 
     │ String7  Int64              
─────┼─────────────────────────────
   1 │ week_01                 168
   2 │ week_02                 168
   3 │ week_03                 168
   4 │ week_04                 168
   5 │ week_05                  48

In [1]:
# Load the data about the bottling time for each beer
bottlingTime = CSV.read("$file_directory/bottlingtime.csv", DataFrame)
println("Number of beers: $(nrow(bottlingTime))")
println("Bottling time per beer:")
println(bottlingTime)

Number of beers: 6
Bottling time per beer:
6×2 DataFrame
 Row │ beer_type   bottling_time 
     │ String15    Float64       
─────┼───────────────────────────
   1 │ Pilsener          0.00222
   2 │ Blonde_Ale        0.00111
   3 │ Amber_Ale         0.00139
   4 │ Brown_Ale         0.00222
   5 │ Porter            0.00167
   6 │ Stout             0.00111

In [1]:
# Load the data about the setup time for each beer
setupTime = CSV.read("$file_directory/setuptime.csv", DataFrame)
println("Setup time per beer:")
println(setupTime)

Setup time per beer:
6×2 DataFrame
 Row │ beer_type   setup_time 
     │ String15    Int64      
─────┼────────────────────────
   1 │ Pilsener            10
   2 │ Blonde_Ale          11
   3 │ Amber_Ale            8
   4 │ Brown_Ale            8
   5 │ Porter              11
   6 │ Stout                9

In [1]:
# Load the data about the weekly demand for each beer
demandCustomers = CSV.read("$file_directory/demand.csv", DataFrame)
println("First 5 rows of demand per beer:")
println(demandCustomers[1:5, :])

First 5 rows of demand per beer:
5×3 DataFrame
 Row │ beer_types  variable  value 
     │ String15    String7   Int64 
─────┼─────────────────────────────
   1 │ Pilsener    week_01    3853
   2 │ Blonde_Ale  week_01    8372
   3 │ Amber_Ale   week_01   16822
   4 │ Brown_Ale   week_01   13880
   5 │ Porter      week_01   10642

Next, we define the model instance for the CLSP.

In [0]:
# Prepare the model instance
lotsizeModel = Model(HiGHS.Optimizer)
set_attribute(lotsizeModel, "presolve", "on")
set_time_limit_sec(lotsizeModel, 60.0)

Consider in your implementation, that each hour of setup is associated
with a cost of 1000 Euros, and the inventory holding cost for unsold
bottles at the end of each period is 0.1 Euro per bottle. Implement
**both parameters** for the cost of setup and the inventory holding cost
in the model.

In [0]:
# Define the cost of setup and the inventory holding cost as parameters
# YOUR CODE BELOW


Next, you need to prepare the given data for the model.

In [0]:
# Prepare the data for the model
# YOUR CODE BELOW


Now, create your variables. Please name them `productBottled` for the
binary variable, `productQuantity` for the production quantity and
`WarehouseStockPeriodEnd` for the warehouse stock at the end of each
period. We will use these names later in the code to plot the results.

In [0]:
# Create your variables below
# YOUR CODE BELOW


Next, define the objective function.

In [0]:
# Define the objective function below
# YOUR CODE BELOW


Now, define all necessary constraints for the model.

In [0]:
# Define all necessary constraints for the model
# YOUR CODE BELOW


Finally, implement the solve statement for your model instance.

In [0]:
# Implement the solve statement for your model instance
# YOUR CODE BELOW


If you correctly named your variables as stated above, the following
should create the production and warehouse plots for you.

In [0]:
productionResults = DataFrame(
    period = String[],
    product = String[],
    productBottled = Bool[],
    productQuantity=Int[],
    WarehouseStockPeriodEnd=Int[]
    )
for i in axes(setupTime,1)
    for t in axes(availableTime,1)
        push!(productionResults,(
            period = t,
            product = i,
            productBottled = value(Yit[i,t])>0.5 ? true : false,
            productQuantity = ceil(Int,value(Xit[i,t])),
            WarehouseStockPeriodEnd = ceil(Int,value(Wit[i,t])),
        ))
    end
end

sort!(productionResults,[:period, :product])

p = groupedbar(
    productionResults.period, 
    productionResults.productQuantity, 
    group=productionResults.product, 
    ylabel="Production Quantity", 
    xlabel="Period", 
    size=(1200,600),
    palette = :Pastel1_6, 
    legend=:outertopright
)

p = groupedbar(
    productionResults.period, 
    productionResults.WarehouseStockPeriodEnd, 
    group=productionResults.product, 
    ylabel="Warehouse", xlabel="Period", size=(1200,600),
    palette = :Pastel1_6, legend=:outertopright
)

# 2. Initial Warehouse Stock

The model currently sets the initial warehouse stock levels without any
restrictions. Modify your model to incorporate an initial stock for
**all types of beer of zero** at the beginning of the initial planning
period.

You can do this in the existing model above.

# 3. Scheduled Repair

Unfortunately, the bottling plant has to undergo maintenance in periods
`"week_10"` and `"week_11"`. Extend your model to prevent any production
in those two periods.

To solve this task, you can simply extend the previous model by these
additional constraints.

# 4. Production Schedule Analysis

Analyze the production schedule outlined in section 2 of this tutorial.
Is the workload **distributed evenly** across all time periods? Provide
a rationale for your assessment and a plot of the aggregated production
quantities over all beer sorts.

Answer your assessment in the following markdown cell.

``` {markdown}
# YOUR REASONING BELOW
```

Next, create the plot of the aggregated production quantities over all
beer sorts in the following cell.

In [0]:
# YOUR PLOT BELOW


# 5. Ending Inventory Analysis

Based on the production data from the final period in section 2,
calculate the ending inventory levels for each type of beer. Discuss any
significant findings.

Compute the ending inventory levels for each type of beer in the
following cell.

In [0]:
# YOUR CODE BELOW


Discuss any significant findings in the following markdown cell.

``` {markdown}
# YOUR ANSWER BELOW
```

# 6. Biannual Bottling Strategy

Reflecting on a scenario where the company schedules its bottling
operations **biannually** using the current method: identify and discuss
potential pitfalls of this strategy. Offer at least one actionable
suggestion for enhancing the efficiency or effectiveness of the
production planning process.

Your answer goes here.

``` {markdown}
# YOUR ANSWER BELOW
```